# Experiment 5: ML algorithms with hyperparameter tuning

TF-IDF trigrams (1,000 features) + SMOTE on training data, 30 Optuna trials per algorithm.

Changes from the original notebook:
- **No tuning on the test set.** Optuna scores trials on a validation split carved out of the training
  data; the test set is used once, for the final score of each tuned model.
- **All algorithms actually run.** The original only ran XGBoost, and that run was interrupted at trial 13.
  The video also mentions decision trees, KNN and Naive Bayes, so they are included here.
  LightGBM gets its own detailed notebook (experiment 6).

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000'))
EXPERIMENT = 'Exp 5 - ML Algos with HP Tuning'
mlflow.set_experiment(EXPERIMENT)
BATCH = datetime.now().strftime('%Y%m%d-%H%M%S')
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 30
NGRAM_RANGE = (1, 3)
MAX_FEATURES = 1000

In [ ]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])

# XGBoost needs labels 0..n-1, so -1 (negative) becomes 2 for training; reports map it back to -1
TO_MODEL = {-1: 2, 0: 0, 1: 1}
TO_LABEL = {v: k for k, v in TO_MODEL.items()}
df['category'] = df['category'].map(TO_MODEL)

X_train_text, X_test_text, y_train_raw, y_test = train_test_split(
    df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)
X_fit_text, X_val_text, y_fit_raw, y_val = train_test_split(
    X_train_text, y_train_raw, test_size=0.2, random_state=42, stratify=y_train_raw
)


def build_features(train_text, train_labels, *eval_texts):
    """Fit TF-IDF on the training text, oversample it with SMOTE, and transform the evaluation text."""
    vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES)
    X = vectorizer.fit_transform(train_text)
    X, y = SMOTE(random_state=42).fit_resample(X, train_labels)
    return (X, y, *[vectorizer.transform(t) for t in eval_texts])


# Tuning data: fit split scored on the validation split
X_fit, y_fit, X_val = build_features(X_fit_text, y_fit_raw, X_val_text)
# Final data: the whole training split scored on the test split
X_train, y_train, X_test = build_features(X_train_text, y_train_raw, X_test_text)
X_fit.shape, X_val.shape, X_train.shape, X_test.shape

In [ ]:
def log_evaluation(y_true, y_pred, title):
    """Log accuracy, per-class metrics and a confusion matrix to the active MLflow run."""
    mlflow.log_metric('accuracy', accuracy_score(y_true, y_pred))
    for label, metrics in classification_report(y_true, y_pred, output_dict=True).items():
        if isinstance(metrics, dict):
            mlflow.log_metrics({f'{label}_{name}': value for name, value in metrics.items()})

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred, labels=[-1, 0, 1]), annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[-1, 0, 1], yticklabels=[-1, 0, 1])
    ax.set(xlabel='Predicted', ylabel='Actual', title=f'Confusion Matrix: {title}')
    mlflow.log_figure(fig, 'confusion_matrix.png')
    plt.close(fig)


# Each entry builds a model from an Optuna trial; FixedTrial replays the best parameters later
SEARCH_SPACES = {
    'LogisticRegression': lambda t: LogisticRegression(
        C=t.suggest_float('C', 1e-4, 10.0, log=True), max_iter=1000),
    'MultinomialNB': lambda t: MultinomialNB(
        alpha=t.suggest_float('alpha', 1e-4, 1.0, log=True)),
    'DecisionTree': lambda t: DecisionTreeClassifier(
        max_depth=t.suggest_int('max_depth', 3, 50),
        min_samples_split=t.suggest_int('min_samples_split', 2, 20),
        criterion=t.suggest_categorical('criterion', ['gini', 'entropy']),
        random_state=42),
    'KNN': lambda t: KNeighborsClassifier(
        n_neighbors=t.suggest_int('n_neighbors', 3, 30),
        weights=t.suggest_categorical('weights', ['uniform', 'distance']),
        metric=t.suggest_categorical('metric', ['euclidean', 'cosine']),
        n_jobs=-1),
    'RandomForest': lambda t: RandomForestClassifier(
        n_estimators=t.suggest_int('n_estimators', 50, 300),
        max_depth=t.suggest_int('max_depth', 3, 20),
        min_samples_split=t.suggest_int('min_samples_split', 2, 20),
        random_state=42, n_jobs=-1),
    'XGBoost': lambda t: XGBClassifier(
        n_estimators=t.suggest_int('n_estimators', 50, 300),
        learning_rate=t.suggest_float('learning_rate', 1e-4, 1e-1, log=True),
        max_depth=t.suggest_int('max_depth', 3, 10),
        tree_method='hist', random_state=42, n_jobs=-1),
}

In [ ]:
def tune_and_log(name, build_model):
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda t: accuracy_score(y_val, build_model(t).fit(X_fit, y_fit).predict(X_val)), n_trials=N_TRIALS)

    best_model = build_model(optuna.trial.FixedTrial(study.best_params))
    best_model.fit(X_train, y_train)
    y_pred = pd.Series(best_model.predict(X_test)).map(TO_LABEL)

    with mlflow.start_run(run_name=f'{name}_SMOTE_TFIDF_Trigrams'):
        mlflow.set_tags({'experiment_type': 'algorithm_comparison', 'batch': BATCH})
        mlflow.log_params({'algo_name': name, 'n_trials': N_TRIALS, **study.best_params})
        mlflow.log_metric('val_accuracy', study.best_value)
        log_evaluation(y_test.map(TO_LABEL), y_pred, name)
        mlflow.sklearn.log_model(best_model, f'{name}_model')

    test_accuracy = accuracy_score(y_test.map(TO_LABEL), y_pred)
    print(f'{name:<20} val={study.best_value:.4f}  test={test_accuracy:.4f}  {study.best_params}')


for name, build_model in SEARCH_SPACES.items():
    tune_and_log(name, build_model)

In [ ]:
runs = mlflow.search_runs(experiment_names=[EXPERIMENT], filter_string=f"tags.batch = '{BATCH}'")
columns = {
    'params.algo_name': 'algorithm',
    'metrics.val_accuracy': 'val_accuracy',
    'metrics.accuracy': 'test_accuracy',
    'metrics.-1_recall': 'neg_recall',
    'metrics.macro avg_f1-score': 'macro_f1',
}
runs[list(columns)].rename(columns=columns).sort_values('test_accuracy', ascending=False).round(4)

LightGBM is tuned in detail in experiment 6; the video ultimately picks it as the final model.